# Gravedad de los siniestros de transporte en Colombia (2015-2024)
## Pipeline integral de Big Data con Machine Learning — Proyecto final

**Gerencia y almacenamiento en Big Data** · Autor: **Juan Rojas** · Modalidad individual

**Fuentes:** 4 datasets oficiales de datos.gov.co — Instituto Nacional de Medicina Legal y Ciencias Forenses (lesiones y muertes por eventos de transporte) y Policía Nacional (lesiones y homicidios culposos en accidente de tránsito).

**Video de sustentación:** _[pendiente: pegar aquí el enlace no listado]_

**Repositorio:** https://github.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFinal

| Sección | Fase CRISP-DM | Estado |
|---|---|---|
| 1. Business Understanding y selección del dataset | Business Understanding | ✅ |
| 2. Decisión de arquitectura: pandas o Spark | — | ✅ |
| 3. ETL: exploración, calidad, unificación de 4 fuentes, 4 técnicas | Data Understanding + Data Preparation | ✅ |
| 4. Análisis exploratorio y visualizaciones | Data Understanding | ⏳ |
| 5. Machine Learning | Modeling + Evaluation | ⏳ |
| 6. Insights en MongoDB | Deployment | ⏳ |
| 7. Reporte PDF | Deployment | ⏳ |

In [1]:
import sys
import json
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import config
import etl

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)
print("Raíz del proyecto:", RAIZ)
print("pandas", pd.__version__, "| numpy", np.__version__, "| Python", platform.python_version())

Raíz del proyecto: /home/claude/Rojas_Juan_BigData_ProyectoFinal
pandas 3.0.5 | numpy 2.5.3 | Python 3.13.18


---
# 1. Business Understanding y selección del dataset

## 1.1 Contexto
Los siniestros de transporte son una de las principales causas de muerte violenta en Colombia. Cada víctima queda registrada, al menos, por dos instituciones con propósitos distintos:

- **Medicina Legal (INMLCF)** registra a cada víctima que pasa por **examen médico-legal**: lesionados (con sus días de incapacidad) y fallecidos (necropsia). Es un registro **forense y muy detallado** (~38 variables: víctima, vehículo, tipo de accidente, objeto de colisión, causa, lugar, hora).
- **Policía Nacional** registra las **denuncias** por lesiones culposas (art. 120 del Código Penal) y homicidios culposos (art. 109) en accidente de tránsito. Es un registro **judicial con pocas variables** (9) pero con **fecha exacta** y gran cobertura.

Integrar ambas visiones permite responder qué hace más grave un siniestro y, a la vez, dimensionar el fenómeno completo.

## 1.2 Descripción de las fuentes

In [2]:
# Descarga lo que falte desde el GitHub del proyecto (respaldo: datos.gov.co) y registra metadatos.
# Si los 4 CSV ya están en data/raw/, no descarga nada y solo registra sus metadatos.
import descargar_datos
metadata = descargar_datos.descargar_todo()

inicio = time.perf_counter()
crudos = etl.cargar_fuentes()
t_carga_texto = time.perf_counter() - inicio

perfil = etl.perfil_fuentes(crudos)
perfil["pagina"] = [config.FUENTES[n]["pagina"] for n in perfil.index]
perfil["url_descarga"] = [metadata[n]["url_descarga"] for n in perfil.index]
perfil["fecha_descarga"] = [metadata[n]["fecha_descarga"] for n in perfil.index]
display(perfil)
print(f"Total: {perfil['filas'].sum():,} filas | {perfil['disco_mb'].sum():,.1f} MB en disco | "
      f"{perfil['memoria_mb'].sum():,.1f} MB en memoria (todo como texto)")

→ inml_lesiones: intentando fuente github


    1.0 MB

    2.1 MB

    3.1 MB

    4.2 MB

    5.2 MB

... (22 outputs left out)

✓ policia_lesiones: 1,381,416 filas, 191.1 MB
→ policia_homicidios_transito: intentando fuente github
    0.9 MB


✓ policia_homicidios_transito: 98,430 filas, 14.8 MB
Metadatos en data/raw/metadata_descarga.json


,entidad,id_datos_gov,filas,columnas,disco_mb,memoria_mb,periodo,pagina,url_descarga,fecha_descarga
fuente,,,,,,,,,,
inml_lesiones,Instituto Nacional de Medicina Legal y Ciencias Forenses (INMLCF),ezhf-hscf,342796,39,238.4,305.3,2015-2024,https://www.datos.gov.co/d/ezhf-hscf,https://raw.githubusercontent.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFin...,2026-11-16T02:30:24
inml_muertes,Instituto Nacional de Medicina Legal y Ciencias Forenses (INMLCF),s65h-7665,73403,38,47.5,61.5,2015-2024,https://www.datos.gov.co/d/s65h-7665,https://raw.githubusercontent.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFin...,2026-11-16T02:30:24
policia_lesiones,Dirección General de la Policía Nacional (DIPON),72sg-cybi,1381416,9,191.1,253.6,2010-2026,https://www.datos.gov.co/d/72sg-cybi,https://raw.githubusercontent.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFin...,2026-11-16T02:30:25
policia_homicidios_transito,Dirección General de la Policía Nacional (DIPON),ha6j-pa2r,98430,9,14.8,19.3,2010-2026,https://www.datos.gov.co/d/ha6j-pa2r,https://raw.githubusercontent.com/ingjuancrojasl-crypto/Rojas_Juan_BigData_ProyectoFin...,2026-11-16T02:30:25


Total: 1,896,045 filas | 491.8 MB en disco | 639.7 MB en memoria (todo como texto)


**¿Por qué estas fuentes?**
- Son **reales, públicas y oficiales**, de dos instituciones independientes.
- **Volumen:** 1,9 millones de filas crudas. Tras quedarse con el tránsito y el periodo común 2015-2024 se obtienen **767.313 registros**, que corresponden a **960.623 víctimas** (algunas filas de la Policía agrupan varias víctimas). Supera los 500.000 registros recomendados.
- **No vienen limpias:** cambios de catálogo entre años, variantes de escritura, formatos de código distintos, duplicados, valores incoherentes y, sobre todo, **diligenciamiento desigual entre lesiones y muertes**, que obliga a cuidar el modelo frente a la fuga de información (sección 3).
- **Útiles para ML:** permiten construir una variable objetivo de **gravedad** con tres niveles a partir de los días de incapacidad médico-legal y del desenlace fatal.

## 1.3 Pregunta de negocio
> **¿Qué características de la víctima, del vehículo y del contexto se asocian con la gravedad de un siniestro de transporte (lesión leve, lesión grave o muerte), y cuáles son las causas más frecuentes en lesionados y fallecidos?**

## 1.4 Preguntas específicas
1. ¿Cómo evolucionaron las lesiones y las muertes entre 2015 y 2024, y qué cambió con la pandemia de 2020?
2. ¿Qué perfil de víctima (sexo, edad, conductor/pasajero/peatón) concentra los lesionados y cuál los fallecidos?
3. ¿Qué vehículos, tipos de accidente y objetos de colisión tienen mayor proporción de muertes?
4. ¿Cuáles son las causas más frecuentes en lesionados y en fallecidos?
5. ¿Dónde (región, zona urbana o rural) y cuándo (día, franja horaria) son más graves los siniestros?
6. ¿Coinciden Medicina Legal y la Policía en el número de víctimas por departamento y año?

## 1.5 Tarea de Machine Learning y criterio de éxito
- **Tarea:** **clasificación multiclase** de `gravedad` ∈ {Lesión leve, Lesión grave, Muerte}.
  - Lesión leve: 0 a 30 días de incapacidad médico-legal · Lesión grave: más de 30 días · Muerte: registro de necropsia.
- **Datos:** víctimas de Medicina Legal con gravedad conocida (~399.000). La Policía no registra la gravedad de las lesiones, por eso no entra como datos de entrenamiento; se usa como **contexto municipal** (víctimas del año anterior) y en el análisis.
- **Predictores:** solo variables **registradas de forma comparable** en lesiones y muertes (víctima, vehículo, accidente, objeto, zona, día, mes, región). Se excluyen hora, causa, actividad y escolaridad como predictores porque su ausencia delata el desenlace (sección 3.2).
- **Validación temporal:** entrenamiento 2015-2022, prueba 2023-2024.
- **Línea base:** clasificador de clase mayoritaria y clasificador estratificado.
- **Criterio de éxito:** que el mejor modelo supere el **F1 macro** de la mejor línea base en al menos **10 puntos porcentuales**, con **recall de la clase Muerte ≥ 0,70** (es el error más costoso de pasar por alto).
- **Alcance:** el modelo identifica **factores asociados** a la gravedad; no demuestra causalidad.

---
# 2. Decisión de arquitectura: pandas o Spark

## 2.1 Recursos de la máquina

In [3]:
memoria_total_gb = psutil.virtual_memory().total / 1e9
pd.Series({
    "Sistema operativo": f"{platform.system()} {platform.release()}",
    "Núcleos de CPU (lógicos)": psutil.cpu_count(logical=True),
    "Memoria RAM total": f"{memoria_total_gb:,.1f} GB",
    "Memoria RAM disponible": f"{psutil.virtual_memory().available / 1e9:,.1f} GB",
    "¿Dentro de Docker?": Path("/.dockerenv").exists(),
}, name="valor").to_frame()

,valor
Sistema operativo,Linux 6.18.44-fc-v470
Núcleos de CPU (lógicos),2
Memoria RAM total,8.4 GB
Memoria RAM disponible,6.7 GB
¿Dentro de Docker?,False


## 2.2 Mediciones propias

In [4]:
mediciones = []
mem_texto = perfil["memoria_mb"].sum()
mediciones.append(("Leer los 4 CSV como texto", t_carga_texto, mem_texto))

# a) Tipos optimizados: category para textos repetidos
inicio = time.perf_counter()
optimizados = {n: df.astype("category") for n, df in crudos.items()}
t_opt = time.perf_counter() - inicio
mem_opt = sum(etl.memoria_mb(d) for d in optimizados.values())
mediciones.append(("Convertir a category (4 fuentes)", t_opt, mem_opt))
del optimizados

# b) Lectura por bloques: contar víctimas de tránsito en el archivo más grande sin cargarlo completo
def victimas_transito_por_bloques(tamano=250_000):
    total = 0
    for bloque in pd.read_csv(config.FUENTES["policia_lesiones"]["archivo"], dtype=str, chunksize=tamano,
                              usecols=["DESCRIPCIÓN CONDUCTA", "CANTIDAD"]):
        es_transito = bloque["DESCRIPCIÓN CONDUCTA"].str.contains("EN ACCIDENTE DE TRANSITO", na=False)
        total += bloque.loc[es_transito, "CANTIDAD"].astype(int).sum()
    return total

inicio = time.perf_counter()
victimas_bloques = victimas_transito_por_bloques()
mediciones.append(("Policía lesiones por bloques de 250.000 (2 columnas)", time.perf_counter() - inicio, np.nan))

tabla_mediciones = pd.DataFrame(mediciones, columns=["escenario", "segundos", "memoria_mb"]).round(2)
display(tabla_mediciones)
print(f"Víctimas de tránsito (todos los años) contadas por bloques: {victimas_bloques:,}")
print(f"Reducción de memoria con category: {1 - mem_opt / mem_texto:.1%}")
print(f"Memoria en texto / RAM total: {mem_texto / (memoria_total_gb * 1000):.1%}")

,escenario,segundos,memoria_mb
0,Leer los 4 CSV como texto,5.65,639.7
1,Convertir a category (4 fuentes),0.78,42.5
2,Policía lesiones por bloques de 250.000 (2 columnas),1.37,NaN


Víctimas de tránsito (todos los años) contadas por bloques: 754,670
Reducción de memoria con category: 93.4%
Memoria en texto / RAM total: 7.6%


## 2.3 Decisión: **pandas**

| Criterio (Actividad 4) | Medición propia | Implicación |
|---|---|---|
| Volumen | 1,9 M filas crudas en 4 archivos; ~490 MB en disco | Mediano: cabe en una sola máquina |
| Memoria | ~640 MB como texto (~8 % de la RAM) → ~43 MB con `category` (−93 %); las dos bases analíticas en Parquet ocupan ~42 MB en memoria | Una fracción pequeña de la RAM |
| Tiempo | Lectura de los 4 CSV en ~5 s; filtrado por bloques del archivo más grande en ~1 s; ETL completo en ~20 segundos | No hay cuello de botella que justifique distribuir |
| Infraestructura | Un nodo (portátil o contenedor) | Spark añadiría JVM, SparkSession y sobrecarga de planificación sin beneficio |
| Crecimiento | Las fuentes crecen ~100.000 víctimas/año | Seguiría siendo manejable con pandas por años |

Técnicas aplicadas para manejar el volumen con pandas:
1. **Lectura como texto** (`dtype=str`) para no perder ceros iniciales de los códigos DANE, y luego **tipos `category`** y enteros pequeños.
2. **Lectura por bloques** (`chunksize`) para filtrar el archivo más grande sin cargarlo completo, si el volumen creciera.
3. **Parquet** columnar comprimido para la base analítica, la unificada y los cubos OLAP: el dashboard lee Parquet, nunca los CSV.

Spark sería la elección si se integraran, por ejemplo, todos los registros de lesiones de causa externa de varios años (decenas de millones de filas) o datos que no caben en memoria.

---
# 3. ETL con CRISP-DM

## 3.1 Data Understanding: exploración inicial (antes de limpiar)
### Estructura de cada fuente

In [5]:
for nombre, df in crudos.items():
    print(f"\n=== {nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    display(df.head(3))


=== inml_lesiones: 342,796 filas × 39 columnas


,ID,Año del hecho,Sexo de la victima,Grupo de edad quinquenal,Grupo Mayor Menor de Edad,Grupo de Edad judicial,Ciclo Vital,País de Nacimiento,Escolaridad,Estado Civil,Tipo de Discapacidad,Pertenencia Étnica,Orientación Sexual,Identidad de Género,Transgénero,Pertenencia Grupal,Mes del hecho,Dia del hecho,Rango de Hora del Hecho X 3 Horas,Código Dane Municipio,Municipio del hecho DANE,Departamento del hecho DANE,Código Dane Departamento,Localidad del Hecho,Zona del Hecho,Escenario del Hecho,Actividad Durante el Hecho,Circunstancia del Hecho Detallada,Contexto del Hecho,Mecanismo Causal de la Lesión no Fatal,Diagnostico Topográfico de la Lesión no Fatal,Condición de la Víctima (AT),Medio de Desplazamiento o Transporte,Servicio del Vehículo,Clase o Tipo de Accidente,Objeto de Colisión,Servicio del Objeto de Colisión,Días de Incapacidad Medicolegal,Pueblo Indígena
0,1,2015,Hombre,(20 a 24),b) Mayor de Edad (>18 Años),(20 a 24),(18 a 28) Juventud,Colombia,Básica secundaria,Unión libre,Ninguna,Raizal,No Sabe / No Informa,No Sabe / No Informa,No Sabe / No Informa,Sin información,Septiembre,viernes,(03:00 a 05:59),88001,San Andrés,"Archipiélago de San Andrés, Providencia y Santa Catalina",88,Sin información,Cabecera municipal,"Vía pública, calle, carretera",Actividades de desplazamiento de un lugar a otro.,Adelantar invadiendo carril de sentido contrario,7 Lesiones por Eventos de Transporte,Contundente,Politraumatismo,Conductor,Motocicleta,Particular,Caída del ocupante,Sin información,No aplica,1 a 30,No había sido implementada
1,2,2015,Hombre,(55 a 59),b) Mayor de Edad (>18 Años),(55 a 59),(29 a 59) Adultez,Colombia,Preescolar,Soltero (a),Ninguna,Sin pertenencia étnica,No Sabe / No Informa,No Sabe / No Informa,No Sabe / No Informa,Sin información,Noviembre,sábado,(15:00 a 17:59),52001,Pasto,Nariño,52,Sin información,Cabecera municipal,"Vía pública, calle, carretera",Actividades de desplazamiento de un lugar a otro.,Exceso de velocidad,7 Lesiones por Eventos de Transporte,Biodinámico,Politraumatismo,Peatón,No aplica,No aplica,Atropello,Motocicleta,Particular,1 a 30,No había sido implementada
2,3,2015,Hombre,(30 a 34),b) Mayor de Edad (>18 Años),(29 a 34),(29 a 59) Adultez,Colombia,Profesional,Casado (a),Ninguna,Sin información,No Sabe / No Informa,No Sabe / No Informa,No Sabe / No Informa,Sin información,Abril,jueves,(15:00 a 17:59),8001,Barranquilla,Atlántico,8,Sin información,Cabecera municipal,"Vía pública, calle, carretera",Actividades relacionadas con el trabajo remunerado,Desobedecer señales de tránsito,7 Lesiones por Eventos de Transporte,Contundente,Politraumatismo,Peatón,No aplica,No aplica,Atropello,Automóvil,Público,1 a 30,No había sido implementada



=== inml_muertes: 73,403 filas × 38 columnas


,ID,Año del hecho,Grupo de Edad Quinquenal,Grupo Mayor Menor de Edad,Grupo de Edad judicial,Ciclo Vital,Sexo de la victima,Estado Civil,País de Nacimiento,Escolaridad,Pertenencia Grupal,Pertenencia Étnica,Mes del hecho,Dia del hecho,Rango de Hora del Hecho (X3 Horas),Código Dane Municipio,Municipio del hecho DANE,Departamento del hecho DANE,Código Dane Departamento,Escenario del Hecho,Zona del Hecho,Actividad Durante el Hecho,Circunstancia del Hecho Detallada,Manera de Muerte,Mecanismo Causal de la Lesión Fatal,Diagnostico Topográfico de la Lesión Fatal,Condición de la Víctima (AT),Medio de Desplazamiento o Transporte,Servicio del Vehículo,Clase o Tipo de Accidente de Transporte,Objeto de Colisión,Servicio del Objeto de Colisión,Localidad del Hecho,Ancestro Racial,Pueblo Indígena,Orientación Sexual,Identidad de Género,Transgénero
0,1,2015,(25 a 29),b) Mayor de Edad (>18 Años),(25 a 28),(18 a 28) Juventud,Hombre,Unión libre,Colombia,Básica secundaria,Ninguno,Sin información,enero,domingo,21:00 a 23:59,50223,San Luis de Cubarral,Meta,50,Vía Pública,Parte rural (vereda y campo),Sin información,Sin información,3 Muertes por Eventos de Transporte,Contundente,Trauma craneano,Conductor,Motocicleta,Particular,Caída del ocupante,No aplica,No aplica,Sin información,Mestizo,No había sido implementada,No había sido implementada,No había sido implementada,No había sido implementada
1,2,2015,(40 a 44),b) Mayor de Edad (>18 Años),(40 a 44),(29 a 59) Adultez,Hombre,Unión libre,Colombia,Básica secundaria,Ninguno,Sin información,enero,domingo,21:00 a 23:59,50223,San Luis de Cubarral,Meta,50,Vía Pública,Parte rural (vereda y campo),Sin información,Sin información,3 Muertes por Eventos de Transporte,Contundente,Trauma craneano,Conductor,Motocicleta,Particular,Choque con otro vehículo,Motocicleta,Particular,Sin información,Mestizo,No había sido implementada,No había sido implementada,No había sido implementada,No había sido implementada
2,3,2015,(50 a 54),b) Mayor de Edad (>18 Años),(50 a 54),(29 a 59) Adultez,Hombre,Unión libre,Colombia,Preescolar,Ninguno,Sin información,enero,lunes,00:00 a 02:59,50223,San Luis de Cubarral,Meta,50,Vía Pública,Parte rural (vereda y campo),Sin información,Sin información,3 Muertes por Eventos de Transporte,Contundente,Trauma craneano,Conductor,Motocicleta,Particular,Caída del ocupante,No aplica,No aplica,Sin información,Sin Información,No había sido implementada,No había sido implementada,No había sido implementada,No había sido implementada



=== policia_lesiones: 1,381,416 filas × 9 columnas


,DEPARTAMENTO,MUNICIPIO,CODIGO DANE,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,DESCRIPCIÓN CONDUCTA,CANTIDAD
0,SANTANDER,Bucaramanga (CT),68001000,VEHICULO,23/02/2024,FEMENINO,ADULTOS,ARTÍCULO 120. LESIONES CULPOSAS ( EN ACCIDENTE DE TRANSITO ),1
1,BOYACÁ,Nobsa,15491000,VEHICULO,11/03/2024,MASCULINO,ADULTOS,ARTÍCULO 120. LESIONES CULPOSAS ( EN ACCIDENTE DE TRANSITO ),1
2,SANTANDER,Girón,68307000,VEHICULO,05/03/2024,FEMENINO,ADULTOS,ARTÍCULO 120. LESIONES CULPOSAS ( EN ACCIDENTE DE TRANSITO ),1



=== policia_homicidios_transito: 98,430 filas × 9 columnas


,DEPARTAMENTO,MUNICIPIO,CODIGO DANE,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARÍO,DESCRIPCIÓN CONDUCTA,CANTIDAD
0,CAUCA,Cajibío,19130000,NO REPORTADO,13/02/2014,MASCULINO,ADULTOS,ARTÍCULO 109. HOMICIDIO CULPOSO ( EN ACCIDENTE DE TRÁNSITO),1
1,CAUCA,Santander de Quilichao,19698000,NO REPORTADO,15/02/2014,MASCULINO,MENORES,ARTÍCULO 109. HOMICIDIO CULPOSO ( EN ACCIDENTE DE TRÁNSITO),1
2,GUAJIRA,Riohacha (CT),44001000,NO REPORTADO,15/02/2014,MASCULINO,ADULTOS,ARTÍCULO 109. HOMICIDIO CULPOSO ( EN ACCIDENTE DE TRÁNSITO),1


### Perfil de columnas: Medicina Legal (lesiones) — tipos, nulos y valores únicos

In [6]:
def perfil_columnas(df):
    sin_info = df.apply(etl.pct_sin_informacion)
    return pd.DataFrame({"nulos": df.isna().sum(), "pct_sin_informacion": sin_info.round(1),
                         "valores_unicos": df.nunique()})

perfil_columnas(crudos["inml_lesiones"])

,nulos,pct_sin_informacion,valores_unicos
ID,0,0.0,342796
Año del hecho,0,0.0,10
Sexo de la victima,0,0.0,2
Grupo de edad quinquenal,0,0.0,19
Grupo Mayor Menor de Edad,0,0.0,5
Grupo de Edad judicial,0,0.0,19
Ciclo Vital,0,0.0,7
País de Nacimiento,0,0.1,54
Escolaridad,0,1.9,21
Estado Civil,0,1.3,8


### Perfil de columnas: Policía (lesiones)

In [7]:
perfil_columnas(crudos['policia_lesiones'])

,nulos,pct_sin_informacion,valores_unicos
DEPARTAMENTO,0,0.0,33
MUNICIPIO,12,0.0,1022
CODIGO DANE,0,0.0,3297
ARMAS MEDIOS,0,3.6,45
FECHA HECHO,0,0.0,6056
GENERO,2117,0.2,3
GRUPO ETARIO,810,0.1,3
DESCRIPCIÓN CONDUCTA,0,0.0,12
CANTIDAD,0,0.0,91


### Cobertura temporal por fuente (filas por año)

In [8]:
def anios(df):
    if "Año del hecho" in df:
        return df["Año del hecho"].astype(int)
    return pd.to_datetime(df["FECHA HECHO"], format="%d/%m/%Y").dt.year

cobertura = pd.DataFrame({n: anios(df).value_counts() for n, df in crudos.items()}).sort_index().fillna(0).astype(int)
cobertura

,inml_lesiones,inml_muertes,policia_lesiones,policia_homicidios_transito
2010,0,0,57057,4500
2011,0,0,63406,4295
2012,0,0,75042,4549
2013,0,0,76778,4922
2014,0,0,76441,4835
2015,45806,6884,81327,5344
2016,45256,7280,108619,5609
2017,40114,6754,103916,5261
2018,39537,6879,104593,6017
2019,36832,6892,91254,6053


Medicina Legal cubre exactamente 2015-2024. La Policía cubre 2010 a julio de 2026; el periodo **común** a las 4 fuentes es **2015-2024**. El archivo de lesiones de la Policía incluye además lesiones personales que no son de tránsito.

## 3.2 Tabla de problemas de calidad detectados
Cada fila se mide sobre los datos crudos y se asocia a la técnica ETL que la resuelve.

In [9]:
problemas_calidad = etl.diagnosticar_calidad(crudos)
problemas_calidad

,fuente,categoria,problema,registros_afectados,tratamiento
0,Medicina Legal,Esquema,Columnas equivalentes con nombre distinto entre lesiones y muertes,5,Normalización: renombrar a un esquema común en snake_case
1,Policía lesiones,Alcance,"Filas que NO son de tránsito (lesiones personales, riñas, etc.)",926057,Limpieza: filtrar conducta 'EN ACCIDENTE DE TRANSITO'
2,Policía,Alcance,Filas fuera del periodo común 2015-2024,514111,Limpieza: filtrar al periodo común de las 4 fuentes
3,Medicina Legal,Duplicados,Filas idénticas en las 38-39 columnas (sin contar el ID),266,"Limpieza: eliminar (con tantas variables, coincidir en todo es un registro repetido)"
4,Policía homicidios,Duplicados aparentes,Filas idénticas en las 9 columnas (2010-2026),10959,"Se conservan: con solo 9 variables, dos víctimas distintas del mismo día y municipio s..."
5,Policía lesiones,Granularidad,"Filas de tránsito que agrupan varias víctimas (CANTIDAD > 1, hasta 60)",153360,Normalización: expandir a una fila por víctima
6,Policía,Valores incoherentes,"Medio = escopolamina, arma blanca, arma de fuego o contundente en un accidente de trán...",1231,Limpieza: medio -> 'Sin información' y bandera
7,Policía,Nulos,Registros sin género (vacío o 'NO REPORTA'),2459,Limpieza: 'Sin información'
8,Medicina Legal,Registro desigual,Sin dato de hora del hecho: 6 % en lesiones vs 50 % en muertes,57626,Modelo: no usar como predictor (fuga de información); causas solo con registros con dato
9,Medicina Legal,Registro desigual,Sin dato de causa (circunstancia): 10 % en lesiones vs 66 % en muertes,81482,Modelo: no usar como predictor (fuga de información); causas solo con registros con dato


### El problema más importante: diligenciamiento desigual (riesgo de fuga de información)
Algunas variables tienen muchos más vacíos en el archivo de muertes que en el de lesiones. Si se usan como predictores, el modelo aprende que **"si falta el dato, la víctima murió"**, lo que no es un factor de riesgo sino un efecto de cómo se llenan los formularios.

In [10]:
L = crudos["inml_lesiones"].rename(columns=etl.RENOMBRAR_INML)
M = crudos["inml_muertes"].rename(columns=etl.RENOMBRAR_INML)
filas = []
for col in ["sexo", "grupo_edad_quinquenal", "condicion_victima", "medio_transporte", "clase_accidente",
            "objeto_colision", "zona", "dia_semana", "rango_hora", "circunstancia", "actividad", "escolaridad",
            "estado_civil", "servicio_objeto"]:
    pl = etl.pct_sin_informacion(L[col])
    pm = etl.pct_sin_informacion(M[col])
    filas.append((col, round(pl, 1), round(pm, 1), round(pm - pl, 1)))
desigual = pd.DataFrame(filas, columns=["variable", "% sin dato lesiones", "% sin dato muertes", "diferencia"])
desigual["uso_en_el_modelo"] = np.select([desigual["diferencia"].abs() > 10, desigual["diferencia"].abs() > 5],
                                         ["Excluir como predictor", "Usar con cuidado"], "Usar")
desigual.sort_values("diferencia", ascending=False)

,variable,% sin dato lesiones,% sin dato muertes,diferencia,uso_en_el_modelo
9,circunstancia,9.7,65.8,56.1,Excluir como predictor
8,rango_hora,6.0,50.4,44.4,Excluir como predictor
10,actividad,0.6,33.4,32.8,Excluir como predictor
11,escolaridad,1.9,19.6,17.7,Excluir como predictor
13,servicio_objeto,5.2,21.2,16.0,Excluir como predictor
12,estado_civil,1.3,13.8,12.5,Excluir como predictor
4,clase_accidente,0.4,5.5,5.1,Usar con cuidado
5,objeto_colision,3.8,6.2,2.4,Usar
2,condicion_victima,0.0,2.0,2.0,Usar
0,sexo,0.0,0.0,0.0,Usar


## 3.3 Data Preparation: las 4 técnicas ETL
Cada función devuelve un registro de pasos con conteos **antes / después**.

### Técnica 1 — Limpieza
**Medicina Legal:** esquema común, duplicados, periodo, columnas sin aporte y celdas vacías.

In [11]:
inml_limpio, pasos_limpieza_inml = etl.limpiar_inml(crudos["inml_lesiones"], crudos["inml_muertes"])
pd.DataFrame(pasos_limpieza_inml)

,fuente,tecnica,etapa,antes,despues,diferencia,detalle
0,Medicina Legal,Limpieza,Esquema común (columnas),77,41,-36,Renombrado a snake_case; 5 pares de columnas equivalentes con nombre distinto
1,Medicina Legal,Limpieza,Eliminar duplicados (sin ID),416199,415933,-266,
2,Medicina Legal,Limpieza,Validar periodo,415933,415933,0,
3,Medicina Legal,Limpieza,Excluir columnas constantes o casi vacías (columnas),42,29,-13,Ver COLUMNAS_DESCARTADAS_INML
4,Medicina Legal,Limpieza,Celdas vacías -> 'Sin información',73363,0,-73363,


In [12]:
# Columnas excluidas y motivo
pd.Series(etl.COLUMNAS_DESCARTADAS_INML, name="motivo").to_frame()

,motivo
contexto,constante (un solo valor en el archivo de lesiones)
manera_muerte,constante (un solo valor en el archivo de muertes)
orientacion_sexual,61-83 % sin información; no se registraba en años anteriores
identidad_genero,73-78 % sin información; no se registraba en años anteriores
transgenero,98-99 % sin información
localidad,87-93 % sin información (solo aplica a ciudades con localidades)
tipo_discapacidad,solo existe en el archivo de lesiones
ancestro_racial,solo existe en el archivo de muertes
pueblo_indigena,29-38 % sin información y 'No había sido implementada' en años antiguos
pertenencia_grupal,catálogo de 43-73 valores sin estructura comparable


**Policía:** filtrar solo conductas de tránsito y el periodo común, marcar medios incoherentes y completar nulos.

Los homicidios con filas idénticas **no se eliminan**: con solo 9 variables, dos víctimas distintas del mismo día, municipio, sexo y grupo de edad producen filas iguales, y cada fila de ese archivo es una víctima (`CANTIDAD` = 1 siempre).

In [13]:
policia_limpio, pasos_limpieza_policia = etl.limpiar_policia(crudos["policia_lesiones"],
                                                             crudos["policia_homicidios_transito"])
display(pd.DataFrame(pasos_limpieza_policia))
display(crudos["policia_lesiones"]["DESCRIPCIÓN CONDUCTA"].value_counts().head(6).to_frame("filas"))

,fuente,tecnica,etapa,antes,despues,diferencia,detalle
0,Policía lesiones,Limpieza,Filtrar conducta de tránsito,1381416,455359,-926057,"Se excluyen lesiones personales (art. 111, 119...) y 8.651 'ART. 120' sin mención de t..."
1,Policía,Limpieza,Filtrar periodo 2015-2024,553789,351380,-202409,
2,Policía,Limpieza,Marcar medio incoherente con tránsito,0,1228,1228,"Escopolamina, armas, contundentes -> medio 'Sin información'"
3,Policía,Limpieza,Género/edad vacíos -> 'Sin información',1103,0,-1103,


,filas
DESCRIPCIÓN CONDUCTA,
ARTÍCULO 111. LESIONES PERSONALES,907890
ARTÍCULO 120. LESIONES CULPOSAS ( EN ACCIDENTE DE TRANSITO ),455359
ARTICULO 120 LESIONES CULPOSAS,8651
ARTÍCULO 119. LESIONES PERSONALES ( CIRCUNSTANCIAS DE AGRAVACIÓN),4995
ARTÍCULO 113. DEFORMIDAD,1905
ARTÍCULO 125. LESIONES AL FETO,653


### Técnica 2 — Normalización
- **Textos:** clave sin tildes, mayúsculas, paréntesis ni espacios repetidos; las variantes se unifican en la escritura más frecuente.
- **Catálogos que cambian por año:** *Choque con otro vehículo* (2015-2018) = *Choque* (2019-2024); 4 etiquetas distintas para "cero días de incapacidad".
- **Formatos:** rango horario, mes y día de la semana; códigos DANE a 5 dígitos (municipio) y 2 (departamento); en la Policía, código de 8 dígitos → 5 y nombres oficiales DANE.
- **Unidad de análisis:** en la Policía, una fila con `CANTIDAD = 3` se expande a 3 víctimas, para que todas las fuentes cuenten víctimas.
- **Tipos:** `category` y enteros pequeños.

In [14]:
inml_normal, pasos_normal_inml = etl.normalizar_inml(inml_limpio)
pd.DataFrame(pasos_normal_inml)

,fuente,tecnica,etapa,antes,despues,diferencia,detalle
0,Medicina Legal,Normalización,Unificar escritura en 23 columnas de texto (valores únicos),1686,1485,-201,"Mayúsculas, tildes, espacios, paréntesis"
1,Medicina Legal,Normalización,Clase de accidente: catálogos por año,17,13,-4,'Choque con otro vehículo' (2015-2018) = 'Choque' (2019-2024)
2,Medicina Legal,Normalización,Formato de rango horario,19,9,-10,"'18:00 a 20:59', '(18:00 a 20:59)' -> '(18:00 a 20:59)'"
3,Medicina Legal,Normalización,Zona del hecho,4,4,0,
4,Medicina Legal,Normalización,Códigos DANE a 5 y 2 dígitos,65973,0,-65973,
5,Medicina Legal,Normalización,Tipos optimizados (MB en memoria),264,15,-249,


In [15]:
# Evidencia: cambio de catálogo de 'Clase de accidente' por año (antes de normalizar)
pd.crosstab(L["anio"], L["clase_accidente"])[["Choque", "Choque con otro vehículo"]].T

anio,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
clase_accidente,,,,,,,,,,
Choque,0,766,0,0,26173,11020,18769,24184,24940,23361
Choque con otro vehículo,29805,31425,27974,27865,0,0,0,0,0,0


In [16]:
# Evidencia: variantes de escritura unificadas en el medio de transporte
variantes = (pd.concat([L["medio_transporte"], M["medio_transporte"]]).dropna().drop_duplicates()
             .to_frame("original").assign(clave=lambda d: d["original"].map(etl.clave)))
variantes = variantes.groupby("clave")["original"].agg(list)
variantes[variantes.map(len) > 1].map(lambda v: " | ".join(v)).to_frame("variantes encontradas")

,variantes encontradas
clave,
articulado transporte masivo,Articulado (Transporte masivo) | Articulado (transporte masivo)
automovil,Automóvil | AUTOMÓVIL
"avion , avioneta","Avión, Avioneta | Avión, avioneta"
bus,Bus | BUS
camion - furgon,Camión - furgón | Camión - Furgón
maquinaria agricola,Maquinaria agrícola | Maquinaria Agrícola
microbus,Microbus | Microbús | MIcrobús
motocicleta,Motocicleta | MOTOCICLETA
sin informacion,Sin información | SIN INFORMACIÓN


In [17]:
nombres_dane = inml_normal[["codigo_dane_municipio", "municipio", "codigo_dane_departamento", "departamento"]].astype(str)
policia, pasos_normal_policia = etl.normalizar_policia(policia_limpio, nombres_dane)
display(pd.DataFrame(pasos_normal_policia))
display(policia_limpio[["codigo_dane_policia", "municipio_policia", "genero", "armas_medios", "cantidad"]].head(3))
display(policia[["codigo_dane_municipio", "municipio", "sexo", "medio_grupo", "fecha", "dia_semana"]].head(3))

,fuente,tecnica,etapa,antes,despues,diferencia,detalle
0,Policía,Normalización,Municipio y departamento con nombre oficial DANE (vía código),351380,351354,-26,26 filas con municipio sin víctimas en Medicina Legal: se limpia el nombre de la Policía
1,Policía,Normalización,Expandir CANTIDAD: una fila por víctima,351380,544690,193310,Misma unidad de análisis que Medicina Legal


,codigo_dane_policia,municipio_policia,genero,armas_medios,cantidad
0,68001000,Bucaramanga (CT),FEMENINO,VEHICULO,1
1,15491000,Nobsa,MASCULINO,VEHICULO,1
2,68307000,Girón,FEMENINO,VEHICULO,1


,codigo_dane_municipio,municipio,sexo,medio_grupo,fecha,dia_semana
0,68001,Bucaramanga,Mujer,Vehículo automotor,2024-02-23,Viernes
1,15491,Nobsa,Hombre,Vehículo automotor,2024-03-11,Lunes
2,68307,Girón,Mujer,Vehículo automotor,2024-03-05,Martes


### Técnica 4 — Enriquecimiento
(Se aplica antes de la agregación porque los cubos OLAP usan las variables enriquecidas.)

| Variable nueva | Cómo se construye | Para qué sirve |
|---|---|---|
| `gravedad` | Días de incapacidad (0-30 leve, > 30 grave) y registro de necropsia (muerte) | **Variable objetivo** del modelo |
| `causa_categoria` | 145 descripciones de causa (137 en lesiones, 65 en muertes) → 9 categorías + "Otras" + "Sin información" | Comparar causas entre lesiones y muertes |
| `medio_grupo`, `objeto_grupo` | ~50 tipos de vehículo y objeto → 6 y 10 grupos | Variables interpretables y con suficientes casos |
| `grupo_edad_comun` | Ciclo vital (Medicina Legal) y grupo etario (Policía) → Menores, Adolescentes, Adultos | Comparar las 4 fuentes |
| `adulto_mayor` | Ciclo vital "Más de 60" | Grupo de alto riesgo |
| `franja_horaria`, `fin_de_semana`, `trimestre` | Hora, día y mes | Patrones temporales |
| `region` | Cruce con regiones naturales (código DANE) | Análisis geográfico |
| `escenario_grupo` | Calle urbana / carretera / vía pública | Lugar del hecho |
| `lesionados_policia_anio_ant`, `muertes_policia_anio_ant` | Víctimas registradas por la Policía en el mismo municipio el **año anterior** | Contexto de siniestralidad del municipio sin usar información del mismo año |

In [18]:
policia_extendido, _ = etl.limpiar_policia(crudos["policia_lesiones"], crudos["policia_homicidios_transito"],
                                           anio_inicio=config.ANIO_INICIO - 1)
policia_extendido, _ = etl.normalizar_policia(policia_extendido, nombres_dane)
victimas, pasos_enriquecimiento = etl.enriquecer_inml(inml_normal, policia_extendido)
pd.DataFrame(pasos_enriquecimiento)

,fuente,tecnica,etapa,antes,despues,diferencia,detalle
0,Medicina Legal,Enriquecimiento,Gravedad (leve/grave/muerte) con dato,415933,398950,-16983,Leve: 0-30 días de incapacidad; grave: > 30 días
1,Medicina Legal,Enriquecimiento,Causas: valores distintos -> categorías,80,11,-69,Catálogo aplicado en data/reference/catalogo_causas.csv
2,Medicina Legal,Enriquecimiento,Medio y objeto de colisión agrupados (categorías),105,16,-89,
3,Medicina Legal,Enriquecimiento,Cruce con regiones naturales (filas con región),415933,415857,-76,
4,Medicina Legal,Enriquecimiento,Contexto policial del municipio (año anterior),415933,413204,-2729,Filas cuyo municipio tuvo víctimas policiales el año previo
5,Medicina Legal,Enriquecimiento,Columnas de la base analítica,30,44,14,


In [19]:
# Evidencia: cómo se construye la gravedad a partir de los días de incapacidad
pd.crosstab(victimas["dias_incapacidad"], victimas["gravedad"])

gravedad,Lesión grave,Lesión leve,Muerte,Sin información
dias_incapacidad,,,,
1 a 30,0,171838,0,0
31 a 90,124019,0,0,0
Cero,0,36,0,0
Cero días,0,4645,0,0
Cero días y Sin información,0,0,0,10450
Más de 90,22874,0,0,0
Sin días de incapacidad,0,2175,0,0
Sin información,0,0,73363,6533


In [20]:
# Evidencia: catálogo de causas aplicado (extracto)
catalogo = pd.read_csv(config.ARCHIVO_CATALOGO_CAUSAS)
catalogo.groupby("causa_categoria")["circunstancia"].agg(lambda s: " | ".join(s.head(6))).to_frame("ejemplos")

,ejemplos
causa_categoria,
Condiciones de la vía o el clima,"Dejar obstáculos en la vía | Falta de precaución por niebla, lluvia o humo | Malas con..."
Conducta del pasajero o transporte inseguro,Aprovisionamiento indebido | De pasajero | Descender o subir del vehículo en marcha | ...
Desobedecer señales y normas,Circular por calzadas o carriles destinados a buses y busetas | Conducir en contravía ...
Distracción o fatiga,Cansancio/dormido | Enviar mensajes de texto | Fumar | Hablar por teléfono sin uso de ...
Embriaguez o consumo de sustancias,Cruzar en estado de embriaguez | Embriaguez (Alcohólica y no alcohólica) | Embriaguez ...
Exceso de velocidad,Exceso de velocidad
Fallas mecánicas o del vehículo,Carga sobresaliente sin señales | Falta de mantenimiento mecánico | Falta de señales e...
Imprudencia del peatón,Cruzar en curva | Cruzar en diagonal | Cruzar sin observar | De peatón | Jugar en la v...
Maniobras peligrosas,Adelantar cerrando | Adelantar en curva o pendientes | Adelantar en zona prohibida | A...


## 3.4 Variables derivadas: ¿aportan señal sobre la gravedad?

In [21]:
def gravedad_por(columna, minimo=1000):
    con_gravedad = victimas[victimas["gravedad"] != config.SIN_INFORMACION]
    tabla = pd.crosstab(con_gravedad[columna], con_gravedad["gravedad"], normalize="index").mul(100).round(1)
    tabla["victimas"] = con_gravedad[columna].value_counts()
    return tabla[tabla["victimas"] >= minimo].sort_values("Muerte", ascending=False)

for columna in ["objeto_grupo", "medio_grupo", "condicion_victima", "zona", "grupo_edad_comun", "adulto_mayor"]:
    print(f"\n% por nivel de gravedad según {columna}")
    display(gravedad_por(columna))


% por nivel de gravedad según objeto_grupo


gravedad,Lesión grave,Lesión leve,Muerte,victimas
objeto_grupo,,,,
Objeto fijo,13.1,19.5,67.5,15969
Sin colisión,23.7,39.1,37.2,32153
Sin información,28.5,41.6,29.8,15228
Animal,28.6,43.4,27.9,2419
Vehículo de carga pesada,39.1,39.5,21.4,74750
Bus o buseta,38.4,45.6,16.0,26782
Motocicleta,39.4,45.8,14.8,96949
Automóvil o camioneta,41.1,50.8,8.0,128364
Bicicleta o tracción humana,26.4,67.4,6.2,6206



% por nivel de gravedad según medio_grupo


gravedad,Lesión grave,Lesión leve,Muerte,victimas
medio_grupo,,,,
Peatón (sin vehículo),36.8,38.5,24.7,70450
Motocicleta,39.7,42.7,17.6,231726
Bicicleta y micromovilidad,37.0,47.2,15.8,27208
Sin información,38.1,46.5,15.3,10188
Vehículo automotor,25.4,59.6,15.0,58636



% por nivel de gravedad según condicion_victima


gravedad,Lesión grave,Lesión leve,Muerte,victimas
condicion_victima,,,,
Sin información,0.4,0.9,98.7,1469
Peatón,37.7,40.2,22.1,78774
Conductor,38.9,42.4,18.7,223744
Pasajero,31.8,54.9,13.4,94963


... (2 outputs left out)


% por nivel de gravedad según grupo_edad_comun


gravedad,Lesión grave,Lesión leve,Muerte,victimas
grupo_edad_comun,,,,
Adultos (18 y más),37.5,43.7,18.8,364625
Adolescentes (12-17),31.3,51.5,17.2,20048
Menores (0-11),27.1,62.9,10.0,14241



% por nivel de gravedad según adulto_mayor


gravedad,Lesión grave,Lesión leve,Muerte,victimas
adulto_mayor,,,,
1,35.8,33.2,31.0,49170
0,37.0,46.4,16.6,349780


Las variables derivadas separan bien los niveles de gravedad: por ejemplo, el choque contra **objeto fijo**, la **zona rural** y el **adulto mayor** elevan con fuerza la proporción de muertes.

### Causas más frecuentes en lesionados y fallecidos (solo registros con causa)

In [22]:
con_causa = victimas[~victimas["causa_categoria"].isin([config.SIN_INFORMACION])]
causas = pd.crosstab(con_causa["causa_categoria"], con_causa["desenlace"])
causas["% de lesionados"] = (causas["Lesión"] / causas["Lesión"].sum() * 100).round(1)
causas["% de fallecidos"] = (causas["Muerte"] / causas["Muerte"].sum() * 100).round(1)
display(causas.sort_values("Lesión", ascending=False))
print(f"Registros con causa: lesiones {con_causa['desenlace'].eq('Lesión').mean():.0%} del total con causa; "
      f"solo el {(victimas.query('desenlace == \"Muerte\"')['causa_categoria'] != config.SIN_INFORMACION).mean():.0%} "
      "de las muertes tiene causa registrada.")

desenlace,Lesión,Muerte,% de lesionados,% de fallecidos
causa_categoria,,,,
Maniobras peligrosas,67676,240,21.9,1.0
Desobedecer señales y normas,62520,7743,20.2,30.8
Exceso de velocidad,57356,7722,18.5,30.8
Otras / no especificadas,47036,5614,15.2,22.4
Imprudencia del peatón,35863,249,11.6,1.0
Conducta del pasajero o transporte inseguro,15107,53,4.9,0.2
Embriaguez o consumo de sustancias,9278,1488,3.0,5.9
Fallas mecánicas o del vehículo,6713,1025,2.2,4.1
Condiciones de la vía o el clima,5226,935,1.7,3.7


Registros con causa: lesiones 92% del total con causa; solo el 34% de las muertes tiene causa registrada.


## 3.5 Unificación de las 4 fuentes, cubos OLAP y almacenamiento en Parquet
### Base unificada (columnas comunes)

In [23]:
unificado = etl.unificar(victimas, policia)
print(f"Base unificada: {len(unificado):,} víctimas × {unificado.shape[1]} columnas")
display(pd.crosstab([unificado["fuente"], unificado["archivo_origen"]], unificado["desenlace"], margins=True))

Base unificada: 960,623 víctimas × 16 columnas


desenlace                                     Lesión  Muerte     All
fuente           archivo_origen                                     
Medicina Legal   inml_lesiones                342570       0  342570
                 inml_muertes                      0   73363   73363
Policía Nacional policia_homicidios_transito       0   62599   62599
                 policia_lesiones             482091       0  482091
All                                           824661  135962  960623

### Técnica 3 — Agregación: cubos OLAP

In [24]:
cubos = etl.construir_cubos_olap(unificado, victimas)
for nombre, cubo in cubos.items():
    print(f"{nombre}: {len(cubo):,} filas")
cubos["cubo_geografico"].head()

cubo_geografico: 1,302 filas
cubo_temporal: 3,362 filas
cubo_perfil_victima: 1,194 filas
cubo_gravedad_medicina_legal: 7,849 filas
cubo_causas: 338 filas
cobertura_entre_fuentes: 667 filas


,anio,region,departamento,fuente,desenlace,victimas
0,2015,Amazonía,Amazonas,Medicina Legal,Lesión,63
1,2015,Amazonía,Amazonas,Medicina Legal,Muerte,2
2,2015,Amazonía,Amazonas,Policía Nacional,Lesión,11
3,2015,Amazonía,Caquetá,Medicina Legal,Lesión,101
4,2015,Amazonía,Caquetá,Medicina Legal,Muerte,49


In [25]:
# Operaciones OLAP sobre el cubo geográfico (año × región × departamento × fuente × desenlace)
cubo_geo = cubos["cubo_geografico"]

# Slice: solo muertes
muertes = cubo_geo[cubo_geo["desenlace"] == "Muerte"]
# Dice: muertes de 2023-2024 en la región Caribe
dice = muertes[(muertes["anio"] >= 2023) & (muertes["region"] == "Caribe")]
# Roll-up: de departamento a región
roll_up = muertes.groupby(["region", "fuente"], observed=True)["victimas"].sum().unstack()
# Drill-down: región Andina -> departamentos
drill_down = (muertes[muertes["region"] == "Andina"].groupby(["departamento", "fuente"], observed=True)["victimas"]
              .sum().unstack().sort_values("Medicina Legal", ascending=False))
# Pivot: año × fuente
pivot = muertes.pivot_table(index="anio", columns="fuente", values="victimas", aggfunc="sum")

display(dice.head())
display(roll_up)
display(drill_down.head())
display(pivot)

,anio,region,departamento,fuente,desenlace,victimas
1104,2023,Caribe,Atlántico,Medicina Legal,Muerte,299
1106,2023,Caribe,Atlántico,Policía Nacional,Muerte,242
1108,2023,Caribe,Bolívar,Medicina Legal,Muerte,310
1110,2023,Caribe,Bolívar,Policía Nacional,Muerte,326
1112,2023,Caribe,Cesar,Medicina Legal,Muerte,317


fuente,Medicina Legal,Policía Nacional
region,,
Amazonía,1753.0,1035.0
Andina,38531.0,33482.0
Caribe,14135.0,13194.0
Insular,160.0,143.0
Orinoquía,4848.0,3379.0
Pacífica,13918.0,11366.0
Sin información,18.0,NaN


fuente,Medicina Legal,Policía Nacional
departamento,,
Antioquia,9740.0,9188.0
Cundinamarca,5642.0,4292.0
"Bogotá, D.C.",4441.0,5160.0
Santander,3554.0,2997.0
Tolima,2941.0,2538.0


fuente,Medicina Legal,Policía Nacional
anio,,
2015,6880,5344
2016,7260,5609
2017,6752,5261
2018,6876,6017
2019,6891,6053
2020,5668,5068
2021,7474,6998
2022,8504,7539
2023,8580,7584


### Comparación de cobertura entre fuentes (razón Policía / Medicina Legal)

In [26]:
comparacion = (unificado.groupby(["anio", "desenlace", "fuente"], observed=True).size().unstack("fuente"))
comparacion["razon_policia_vs_medicina_legal"] = (comparacion["Policía Nacional"] / comparacion["Medicina Legal"]).round(2)
comparacion.unstack("desenlace")

fuente    Medicina Legal        Policía Nacional        razon_policia_vs_medicina_legal       
desenlace         Lesión Muerte           Lesión Muerte                          Lesión Muerte
anio                                                                                          
2015               45718   6880            42588   5344                            0.93   0.78
2016               45231   7260            77037   5609                            1.70   0.77
2017               40094   6752            67281   5261                            1.68   0.78
2018               39518   6876            48999   6017                            1.24   0.88
2019               36819   6891            40150   6053                            1.09   0.88
2020               14456   5668            27481   5068                            1.90   0.89
2021               24364   7474            34114   6998                            1.40   0.94
2022               32222   8504            45760   7539                            1.42   0.89
2023               33107   8580            51337   7584                            1.55   0.88
2024               31041   8478            47344   7126                            1.53   0.84

En **muertes**, la Policía registra sistemáticamente menos víctimas que Medicina Legal (entre 0,77 y 0,94 por cada muerte forense): no toda muerte llega a denuncia penal. En **lesiones** ocurre lo contrario desde 2016: la Policía registra entre 1,1 y 1,9 veces más víctimas, porque no todo lesionado que denuncia pasa por examen médico-legal. Ninguna fuente por sí sola dimensiona el fenómeno completo.

In [27]:
# Guardar resultados en data/processed/
problemas_df = problemas_calidad
resumen_etl = {
    "fuentes": perfil.drop(columns=["pagina", "url_descarga", "fecha_descarga"]).reset_index().to_dict(orient="records"),
    "filas_crudas": int(perfil["filas"].sum()),
    "filas_unificadas": len(unificado),
    "filas_medicina_legal": len(victimas),
    "filas_policia_victimas": len(policia),
    "problemas_detectados": problemas_df.to_dict(orient="records"),
    "pasos_etl": (pasos_limpieza_inml + pasos_normal_inml + pasos_limpieza_policia + pasos_normal_policia
                  + pasos_enriquecimiento),
    "agregaciones": {n: len(c) for n, c in cubos.items()},
}
etl.guardar_resultados(victimas, unificado, cubos, resumen_etl)

for ruta in [config.ARCHIVO_DETALLE, config.ARCHIVO_UNIFICADO]:
    df = pd.read_parquet(ruta)
    print(f"{ruta.name}: {ruta.stat().st_size / 1e6:,.1f} MB en disco · {df.shape[0]:,} filas × {df.shape[1]} "
          f"columnas · {etl.memoria_mb(df):,.1f} MB en memoria")
print("Agregaciones:", sorted(p.name for p in config.DIR_AGREGACIONES.glob("*.parquet")))

victimas_medicina_legal.parquet: 9.1 MB en disco · 415,933 filas × 44 columnas · 23.3 MB en memoria
siniestros_unificado.parquet: 4.8 MB en disco · 960,623 filas × 16 columnas · 18.3 MB en memoria
Agregaciones: ['cobertura_entre_fuentes.parquet', 'cubo_causas.parquet', 'cubo_geografico.parquet', 'cubo_gravedad_medicina_legal.parquet', 'cubo_perfil_victima.parquet', 'cubo_temporal.parquet']


### Resumen antes / después del ETL

In [28]:
pd.DataFrame({
    "antes (4 CSV crudos)": [f"{perfil['filas'].sum():,}", "38-39 (Medicina Legal) / 9 (Policía)",
                             f"{perfil['memoria_mb'].sum():,.0f} MB", "Filas mixtas (tránsito y no tránsito, 2010-2026)",
                             f"{pd.concat([L['circunstancia'], M['circunstancia']]).nunique()} descripciones",
                             "No existe"],
    "después": [f"{len(unificado):,} víctimas (unificada) · {len(victimas):,} (Medicina Legal)",
                f"{unificado.shape[1]} comunes / {victimas.shape[1]} en la base analítica",
                f"{etl.memoria_mb(unificado) + etl.memoria_mb(victimas):,.0f} MB",
                "Solo tránsito, 2015-2024, una fila por víctima",
                f"{victimas['causa_categoria'].nunique()} categorías",
                "Leve / Grave / Muerte"],
}, index=["Registros", "Columnas", "Memoria", "Alcance", "Causas", "Gravedad"])

,antes (4 CSV crudos),después
Registros,"1,896,045","960,623 víctimas (unificada) · 415,933 (Medicina Legal)"
Columnas,38-39 (Medicina Legal) / 9 (Policía),16 comunes / 44 en la base analítica
Memoria,640 MB,42 MB
Alcance,"Filas mixtas (tránsito y no tránsito, 2010-2026)","Solo tránsito, 2015-2024, una fila por víctima"
Causas,145 descripciones,11 categorías
Gravedad,No existe,Leve / Grave / Muerte


---
## Próximos pasos
- **Sección 4:** análisis exploratorio con al menos 6 visualizaciones que respondan las preguntas específicas.
- **Sección 5:** modelo de clasificación de gravedad (3 clases): líneas base, al menos 2 modelos, métricas por clase, matriz de confusión, importancia de variables y guardado en `models/`.
- **Secciones 6-7:** documento de análisis en MongoDB y reporte PDF automático.
- Dashboard Streamlit y despliegue con `docker compose up --build`.